In [543]:
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel

import os
from enum import Enum

In [544]:
load_dotenv()

True

In [545]:
LLM_API_URL = os.environ["LLM_API_URL"]
LLM_API_TOKEN = os.environ["LLM_API_TOKEN"]

MODEL = "google/gemma-4-e2b"

In [546]:
client = OpenAI(
    base_url=LLM_API_URL,
    api_key=LLM_API_TOKEN
)

In [547]:
# models = [m.id for m in client.models.list().data]
# models

In [548]:
# response = client.chat.completions.create(
#     model=MODEL,
#     messages=[{"role": "user", "content": "Hi!"}]
# )

# response.choices[0].message.content

# Modélisation du monde

In [549]:
VOID        = 0
PLAYER      = 1
ENNEMY      = 2
GOLD        = 3

SYMBOLS = {VOID: "·", PLAYER: "👤", ENNEMY: "👹", GOLD: "💰"}

In [550]:
initial_map = np.array([
    [0, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 2, 0, 3], # (1, 1) # (1, 4) # (1, 6)
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 3], # (5, 6)
    [0, 0, 0, 0, 0, 0, 0],
])
initial_map

array([[0, 0, 0, 0, 0, 0, 0],
       [0, 1, 0, 0, 2, 0, 3],
       [0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 3],
       [0, 0, 0, 0, 0, 0, 0]])

# Couche de contrat

In [551]:
class Direction(str, Enum):
    HAUT       = "HAUT"
    BAS        = "BAS"
    GAUCHE     = "GAUCHE"
    DROITE     = "DROITE"


# class Confidence(str, Enum):
#     HIGHT       = "HIGHT"
#     MEDIUM      = "MEDIUM"
#     LOW         = "LOW"


class PlayerDecision(BaseModel):
    direction: Direction
    # playerDecision: str
    # confidence: Confidence

# Moteur de perception

In [552]:
def localize(world_map, entity):
    positions = np.argwhere(world_map == entity)
    return positions

In [553]:
# localize(initial_map, ENNEMY)

In [554]:
def compute_distances(entities_positions, reference_pos):
    if (len(entities_positions) == 0):
        return np.array([])
    
    v = entities_positions - reference_pos
    distances = np.linalg.norm(v, axis=1)
 
    return np.round(distances, 2)

In [555]:
golds_distances = compute_distances(localize(initial_map, GOLD), (1, 1))
ennemies_distances = compute_distances(localize(initial_map, ENNEMY), (1, 1))
print(golds_distances, ennemies_distances)

[5.  6.4] [3.]


In [556]:
def perception(world_map):

    player_position = localize(world_map, PLAYER)
    golds_positions = localize(initial_map, GOLD)
    ennemies_positions = localize(initial_map, ENNEMY)

    golds_distances = compute_distances(golds_positions, player_position)
    ennemies_distances = compute_distances(ennemies_positions, player_position)

    return {
        "ennemies_distances": ennemies_distances.tolist(),
        "ennemies_count": len(ennemies_distances),
        "golds_distances": golds_distances.tolist(),
        "golds_count": len(golds_distances),
    }

p = perception(initial_map)
p

{'ennemies_distances': [3.0],
 'ennemies_count': 1,
 'golds_distances': [5.0, 6.4],
 'golds_count': 2}

In [557]:
def show_map(world_map):
    for row in world_map:
        print("\t".join(SYMBOLS.get(cell, "?") for cell in row))
    print('-----------------------------------------------------')

In [558]:
# show_map(initial_map)

# Moteur de déplacement

In [559]:
def allowed_move(world_map: np.ndarray, pos):
    n_rows, n_cols = world_map.shape
    r, c = pos

    if r < 0 or c < 0 or r >= n_rows or c >= n_cols:
        return False
    
    return world_map[r, c] == VOID

In [560]:
def move(world_map: np.ndarray, old_pos, new_pos):
    if not allowed_move(world_map, new_pos):
        return old_pos
    
    entity = world_map[old_pos[0], old_pos[1]]
    world_map[old_pos[0], old_pos[1]] = VOID
    world_map[new_pos[0], new_pos[1]] = entity

    return new_pos

In [561]:
# player_pos = localize(initial_map, PLAYER)[0]
# move(initial_map, player_pos, player_pos + 1)

# Moteur de décision

In [562]:
def decide(player_perception):
    prompt = f"""
    # Contexte
    - Tu es un joueur qui veut maximiser ses gains en or

    # Objectif
    - Trouve le plus court chemin vers l'or

    # Perception
    {player_perception}
    """

    print(prompt)

    response = client.beta.chat.completions.parse(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format=PlayerDecision
    )

    return response.choices[0].message.parsed

In [564]:
p = perception(initial_map)
decision: PlayerDecision = decide(p)
decision.direction.value


    # Contexte
    - Tu es un joueur qui veut maximiser ses gains en or

    # Objectif
    - Trouve le plus court chemin vers l'or

    # Perception
    {'ennemies_distances': [3.0], 'ennemies_count': 1, 'golds_distances': [5.0, 6.4], 'golds_count': 2}
    


'GAUCHE'

# Game loop (simulation)